# 🎥 Video Understanding Pipeline — Kaggle (Multi-Model)

4 vision-language models: **Moondream2**, **SmolVLM**, **Qwen2.5-VL-3B**, **Qwen3-VL-2B**

### ⚠️ Before running:
1. **Accelerator → GPU T4 x2**
2. **Settings → Internet → ON**
3. Run all cells in order

In [ ]:
# ── 1. Install ───────────────────────────────────────────────
# Install transformers from source for Qwen2.5-VL/Qwen3-VL/SmolVLM support
!apt-get update -qq && apt-get install -y -qq libvips-dev > /dev/null 2>&1
!pip install -q git+https://github.com/huggingface/transformers.git accelerate
!pip install -q fastapi uvicorn python-multipart opencv-python-headless \
    Pillow einops pyngrok pyvips qwen_vl_utils
print('✅ All packages installed')

In [ ]:
# ── 2. Verify GPU ────────────────────────────────────────────
import torch
if not torch.cuda.is_available():
    raise RuntimeError('❌ GPU not enabled! Sidebar → Accelerator → GPU T4 x2')
for i in range(torch.cuda.device_count()):
    n = torch.cuda.get_device_name(i)
    v = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}: {n} ({v:.1f} GB)')
print(f'✅ {torch.cuda.device_count()} GPU(s) ready')

In [ ]:
# ── 3. Model Manager (lazy load, one model at a time) ────────
# Fix: Moondream2 + transformers 5.x compatibility
from transformers import modeling_utils as _tmu
# Only patch if not already patched or if method doesn't exist
if not hasattr(_tmu.PreTrainedModel, '_patched_mark'):
    if hasattr(_tmu.PreTrainedModel, 'mark_tied_weights_as_initialized'):
        _orig_mark = _tmu.PreTrainedModel.mark_tied_weights_as_initialized
    else:
        _orig_mark = None

    def _patched_mark(self):
        # Ensure all_tied_weights_keys exists to prevent AttributeError
        if not hasattr(self, 'all_tied_weights_keys'):
            self.all_tied_weights_keys = {}
        if _orig_mark:
            return _orig_mark(self)

    _tmu.PreTrainedModel.mark_tied_weights_as_initialized = _patched_mark
    _tmu.PreTrainedModel._patched_mark = True

import gc, time
from PIL import Image

MODEL_REGISTRY = {
    'moondream2': {'hf_id': 'vikhyatk/moondream2', 'label': 'Moondream2 (~4GB)', 'size_gb': 4},
    'smolvlm':   {'hf_id': 'HuggingFaceTB/SmolVLM-Instruct', 'label': 'SmolVLM (~4GB)', 'size_gb': 4},
    'qwen25vl':  {'hf_id': 'Qwen/Qwen2.5-VL-3B-Instruct', 'label': 'Qwen2.5-VL 3B (~6GB)', 'size_gb': 6},
    'qwen3vl':   {'hf_id': 'Qwen/Qwen3-VL-2B-Instruct', 'label': 'Qwen3-VL 2B (~4GB)', 'size_gb': 4},
}

class ModelManager:
    def __init__(self, device='cuda:0'):
        self.device = device
        self.current = None
        self.model = None
        self.processor = None
    
    def unload(self):
        if self.model is not None:
            del self.model, self.processor
            self.model = self.processor = None
            self.current = None
            gc.collect()
            torch.cuda.empty_cache()
            print('  🗑️ Previous model unloaded')
    
    def load(self, name):
        if name == self.current:
            return
        info = MODEL_REGISTRY[name]
        print(f'⏳ Loading {info["label"]}...')
        self.unload()
        t0 = time.time()
        
        try:
            if name == 'moondream2':
                from transformers import AutoModelForCausalLM
                # Use FP32 for T4 stability as seen in V4/V5
                self.model = AutoModelForCausalLM.from_pretrained(
                    info['hf_id'], revision='2025-01-09', trust_remote_code=True,
                    torch_dtype=torch.float32  # Force FP32 for stability
                ).to(self.device)
                self.processor = None
            
            elif name == 'smolvlm':
                from transformers import AutoProcessor, AutoModelForVision2Seq
                self.processor = AutoProcessor.from_pretrained(info['hf_id'])
                self.model = AutoModelForVision2Seq.from_pretrained(
                    info['hf_id'], torch_dtype=torch.float16,
                    _attn_implementation='eager'
                ).to(self.device)
            
            elif name == 'qwen25vl':
                from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
                self.processor = AutoProcessor.from_pretrained(info['hf_id'])
                # Use single quotes in device_map dict to avoid JSON bugs
                self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
                    info['hf_id'], torch_dtype=torch.float16,
                    device_map={'': self.device})
            
            elif name == 'qwen3vl':
                from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
                self.processor = AutoProcessor.from_pretrained(info['hf_id'])
                self.model = Qwen3VLForConditionalGeneration.from_pretrained(
                    info['hf_id'], torch_dtype=torch.float16,
                    device_map={'': self.device})
            
            self.model.eval()
            self.current = name
            mb = torch.cuda.memory_allocated() / 1e6
            print(f'✅ {info["label"]} loaded in {time.time()-t0:.1f}s ({mb:.0f} MB VRAM)')
        except Exception as e:
            print(f"❌ Failed to load {name}: {e}")
            raise e
    
    def query(self, image: Image.Image, prompt: str) -> str:
        if self.current == 'moondream2':
            ans = self.model.query(image, prompt)
            return ans.get('answer', str(ans)) if isinstance(ans, dict) else str(ans)
        
        elif self.current == 'smolvlm':
            msgs = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': prompt}]}]
            text = self.processor.apply_chat_template(msgs, add_generation_prompt=True)
            inputs = self.processor(text=text, images=[image], return_tensors='pt').to(self.device)
            ids = self.model.generate(**inputs, max_new_tokens=256)
            out = self.processor.batch_decode(ids, skip_special_tokens=True)[0]
            if 'assistant' in out.lower():
                out = out.split('assistant')[-1].strip()
            return out.strip()
        
        elif self.current in ('qwen25vl', 'qwen3vl'):
            # FIX: Use absolute path for Qwen
            import tempfile, os
            with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as tmp:
                image.save(tmp.name)
                tmp_name = tmp.name
            
            try:
                msgs = [{'role': 'user', 'content': [
                    {'type': 'image', 'image': tmp_name}, # Absolute local path
                    {'type': 'text', 'text': prompt}
                ]}]
                
                if self.current == 'qwen25vl':
                    from qwen_vl_utils import process_vision_info
                    text = self.processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
                    img_in, vid_in = process_vision_info(msgs)
                    inputs = self.processor(text=[text], images=img_in, videos=vid_in,
                                           padding=True, return_tensors='pt').to(self.device)
                else:
                    inputs = self.processor.apply_chat_template(
                        msgs, tokenize=True, add_generation_prompt=True,
                        return_dict=True, return_tensors='pt').to(self.device)
                
                ids = self.model.generate(**inputs, max_new_tokens=256)
                trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, ids)]
                out = self.processor.batch_decode(trimmed, skip_special_tokens=True)[0]
                return out.strip()
            finally:
                if os.path.exists(tmp_name): os.remove(tmp_name)
        
        return 'Unknown model'

manager = ModelManager(device='cuda:0')
# Default load
manager.load('moondream2')
print('\\n✅ ModelManager ready.')


In [ ]:
# ── 4. Video Processing ──────────────────────────────────────
import cv2
import numpy as np
from typing import List, Tuple

def extract_frames(path, interval=1.0, max_w=512):
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(int(fps * interval), 1)
    dur = total / fps if total > 0 else 0
    print(f'  Video: {dur:.1f}s, {fps:.0f}fps, every {interval}s')
    frames, idx = [], 0
    while True:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, f = cap.read()
        if not ok: break
        ts = idx / fps
        h, w = f.shape[:2]
        if w > max_w:
            s = max_w / w
            f = cv2.resize(f, (max_w, int(h * s)))
        frames.append((ts, Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))))
        idx += step
    cap.release()
    print(f'  Extracted {len(frames)} frames')
    return frames

def analyze_video(path, prompt, model_name=None, interval=1.0):
    if model_name and model_name != manager.current:
        manager.load(model_name)
    frames = extract_frames(path, interval)
    results = []
    for i, (ts, img) in enumerate(frames):
        mm, ss = int(ts // 60), int(ts % 60)
        tstamp = f'{mm:02d}:{ss:02d}'
        print(f'  Frame {i+1}/{len(frames)} @ {tstamp}', end=' ')
        desc = manager.query(img, prompt)
        print(f'→ {desc[:80]}' + ('...' if len(desc) > 80 else ''))
        results.append({'timestamp': tstamp, 'seconds': round(ts,2),
                       'description': desc.strip(), 'frame_index': i+1})
        torch.cuda.empty_cache()
    return results

print('✅ Video utilities ready')

---
## Option A: Analyze in Notebook
Upload via Kaggle's `+Add Data` sidebar.

In [ ]:
# ── 5A. Notebook Analysis ────────────────────────────────────
import os, time as _time, glob
vids = [f for f in glob.glob('/kaggle/input/**/*.*', recursive=True)
        if os.path.splitext(f)[1].lower() in {'.mp4','.webm','.avi','.mov','.mkv'}]
if vids:
    for i,v in enumerate(vids): print(f'  [{i}] {v} ({os.path.getsize(v)/1e6:.1f}MB)')
    # ── Settings ──
    VIDEO_INDEX = 0
    PROMPT = 'Describe what is happening in this frame in detail.'
    MODEL = 'moondream2'  # moondream2 | smolvlm | qwen25vl | qwen3vl
    # ──────────────
    t0 = _time.time()
    results = analyze_video(vids[VIDEO_INDEX], PROMPT, MODEL)
    print(f'\n📊 Done in {_time.time()-t0:.1f}s')
    for r in results: print(f"[{r['timestamp']}] {r['description']}")
else:
    print('📂 No videos in /kaggle/input/ — use +Add Data or Option B')

---
## Option B: Web UI (ngrok)

In [ ]:
# ── 5B-1. ngrok Token ────────────────────────────────────────
NGROK_AUTH_TOKEN = '39hGJGWCOEVza1i8Uu0BGvRIYWd_7mckVHomfD8WxbRCfa8BC'  # ← paste here
if NGROK_AUTH_TOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print('✅ ngrok configured')
else:
    print('⚠️ Set NGROK_AUTH_TOKEN above')

In [ ]:
# ── 5B-2. UI Template ────────────────────────────────────────
FULL_HTML = r"""
<!DOCTYPE html><html lang="en"><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1.0">
<title>VideoAI Multi-Model</title>
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap" rel="stylesheet">
<style>
*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
:root{--bg:#0a0a0f;--card:rgba(18,18,28,.75);--card-h:rgba(25,25,40,.85);--border:rgba(255,255,255,.06);--border-h:rgba(255,255,255,.12);--text:#e8e8f0;--text2:#8888a8;--muted:#555570;--accent:#7c5cfc;--glow:rgba(124,92,252,.3);--green:#5cf0c8;--danger:#f05868;--radius:16px;--sm:10px;--tr:.25s cubic-bezier(.4,0,.2,1);--font:'Inter',sans-serif}
body{font-family:var(--font);background:var(--bg);color:var(--text);min-height:100vh;line-height:1.6}
.glow{position:fixed;border-radius:50%;filter:blur(120px);opacity:.4;pointer-events:none;z-index:0}
.g1{width:500px;height:500px;background:radial-gradient(circle,var(--accent) 0%,transparent 70%);top:-150px;right:-100px;animation:f1 20s ease-in-out infinite}
.g2{width:400px;height:400px;background:radial-gradient(circle,var(--green) 0%,transparent 70%);bottom:-100px;left:-100px;animation:f2 25s ease-in-out infinite}
@keyframes f1{0%,100%{transform:translate(0,0)}50%{transform:translate(-60px,40px)}}
@keyframes f2{0%,100%{transform:translate(0,0)}50%{transform:translate(40px,-60px)}}
header{position:relative;z-index:10;display:flex;align-items:center;justify-content:space-between;padding:20px 32px;border-bottom:1px solid var(--border);backdrop-filter:blur(20px);background:rgba(10,10,15,.6)}
.logo{display:flex;align-items:center;gap:12px} .logo h1{font-size:22px;font-weight:700} .accent{color:var(--accent)}
.badge{display:flex;align-items:center;gap:8px;padding:6px 14px;border-radius:20px;background:var(--card);border:1px solid var(--border);font-size:13px;color:var(--text2)}
.dot{width:8px;height:8px;border-radius:50%;background:var(--muted);transition:var(--tr)} .dot.on{background:var(--green);box-shadow:0 0 8px var(--green)}
.container{position:relative;z-index:10;display:grid;grid-template-columns:1fr 1fr;gap:24px;max-width:1280px;margin:32px auto;padding:0 24px}
@media(max-width:900px){.container{grid-template-columns:1fr}}
.panel{background:var(--card);border:1px solid var(--border);border-radius:var(--radius);padding:28px;backdrop-filter:blur(24px)} .panel:hover{border-color:var(--border-h)}
.title{font-size:16px;font-weight:600;margin-bottom:20px}
.upload{border:2px dashed var(--border);border-radius:var(--sm);padding:40px 24px;text-align:center;cursor:pointer;transition:var(--tr);margin-bottom:20px}
.upload:hover{border-color:var(--accent);background:rgba(124,92,252,.05)}
.upload-icon{font-size:36px;margin-bottom:8px}
.file-info{display:flex;align-items:center;gap:14px;text-align:left} .file-info .icon{font-size:32px} .file-info .name{font-size:14px;font-weight:500;word-break:break-all} .file-info .size{font-size:12px;color:var(--text2)}
.file-remove{width:32px;height:32px;border:1px solid var(--border);border-radius:8px;background:none;color:var(--text2);cursor:pointer;display:flex;align-items:center;justify-content:center} .file-remove:hover{background:rgba(240,88,104,.1);border-color:var(--danger);color:var(--danger)}
select.model-select{width:100%;padding:12px 16px;border:1px solid var(--border);border-radius:var(--sm);background:rgba(255,255,255,.03);color:var(--text);font-family:var(--font);font-size:14px;margin-bottom:20px;transition:var(--tr);appearance:none;background-image:url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='12' height='12' viewBox='0 0 12 12'%3E%3Cpath fill='%238888a8' d='M6 8L1 3h10z'/%3E%3C/svg%3E");background-repeat:no-repeat;background-position:right 16px center;cursor:pointer}
select.model-select:focus{outline:none;border-color:var(--accent);box-shadow:0 0 0 3px var(--glow)}
select.model-select option{background:#1a1a2e;color:var(--text)}
.prompt{width:100%;padding:12px 16px;border:1px solid var(--border);border-radius:var(--sm);background:rgba(255,255,255,.03);color:var(--text);font-family:var(--font);font-size:14px;resize:vertical;transition:var(--tr);margin-bottom:20px}
.prompt:focus{outline:none;border-color:var(--accent);box-shadow:0 0 0 3px var(--glow)} .prompt::placeholder{color:var(--muted)}
label{display:block;font-size:13px;font-weight:500;color:var(--text2);margin-bottom:8px}
.btn{width:100%;padding:14px;border:none;border-radius:var(--sm);background:linear-gradient(135deg,var(--accent),#9c7cfc);color:#fff;font-size:15px;font-weight:600;cursor:pointer;transition:var(--tr);font-family:var(--font)}
.btn:hover:not(:disabled){transform:translateY(-2px);box-shadow:0 8px 30px var(--glow)} .btn:disabled{opacity:.4;cursor:not-allowed}
.error{margin-top:12px;padding:10px 16px;border-radius:var(--sm);background:rgba(240,88,104,.1);border:1px solid rgba(240,88,104,.2);color:var(--danger);font-size:13px;display:none}
.empty{text-align:center;padding:60px 20px;color:var(--muted)} .empty .icon{font-size:48px;opacity:.5;margin-bottom:16px}
.loading{text-align:center;padding:60px 20px;display:none}
.dots{display:flex;justify-content:center;gap:6px;margin-bottom:20px}
.dots span{width:14px;height:14px;border-radius:50%;background:var(--accent);animation:bounce 1.4s ease-in-out infinite}
.dots span:nth-child(2){animation-delay:.16s} .dots span:nth-child(3){animation-delay:.32s}
@keyframes bounce{0%,80%,100%{transform:scale(.6);opacity:.4}40%{transform:scale(1);opacity:1}}
.meta{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-bottom:14px}
.meta-item{background:rgba(255,255,255,.03);border:1px solid var(--border);border-radius:10px;padding:12px;text-align:center}
.meta-label{display:block;font-size:11px;font-weight:500;color:var(--muted);text-transform:uppercase;margin-bottom:4px}
.meta-val{font-size:16px;font-weight:700;color:var(--green)}
.results-list{display:flex;flex-direction:column;gap:10px;max-height:520px;overflow-y:auto}
.result-card{display:flex;gap:14px;padding:14px 16px;border-radius:var(--sm);background:rgba(255,255,255,.02);border:1px solid var(--border);animation:fadeUp .4s ease-out both}
.result-card:hover{background:var(--card-h);border-color:var(--border-h)}
@keyframes fadeUp{from{opacity:0;transform:translateY(12px)}to{opacity:1;transform:translateY(0)}}
.ts{flex-shrink:0;padding:4px 10px;border-radius:6px;background:var(--glow);color:var(--accent);font-size:12px;font-weight:600;height:fit-content}
.desc{font-size:14px;line-height:1.6}
.model-badge{background:linear-gradient(135deg,#76B900,#4CAF50);color:white;padding:4px 12px;border-radius:12px;font-size:11px;font-weight:600}
.swap-notice{margin-top:12px;padding:10px 16px;border-radius:var(--sm);background:rgba(124,92,252,.1);border:1px solid rgba(124,92,252,.2);color:var(--accent);font-size:13px;display:none;text-align:center}
footer{position:relative;z-index:10;text-align:center;padding:24px;color:var(--muted);font-size:13px;border-top:1px solid var(--border);margin-top:40px} footer strong{color:var(--accent)}
.hidden{display:none!important}
</style></head><body>
<div class="glow g1"></div><div class="glow g2"></div>
<header><div class="logo"><span style="font-size:28px">🎥</span><h1>Video<span class="accent">AI</span></h1></div>
<div style="display:flex;gap:12px;align-items:center"><span class="model-badge" id="modelBadge">MOONDREAM2</span><div class="badge"><span class="dot" id="dot"></span><span id="status">Connecting...</span></div></div></header>
<main class="container">
<section class="panel"><h2 class="title">📤 Input</h2>
<div class="upload" id="dropZone" onclick="document.getElementById('fileInput').click()">
<div id="uploadContent"><div class="upload-icon">☁️</div><p style="font-size:15px;font-weight:500">Drop video or click to upload</p><p style="font-size:13px;color:var(--text2)">MP4, WebM, AVI, MOV — Max 20 MB</p></div>
<div id="fileSelected" class="file-info hidden"></div></div>
<input type="file" id="fileInput" accept=".mp4,.webm,.avi,.mov,.mkv" hidden>
<div><label>🤖 Model</label>
<select class="model-select" id="modelSelect">
<option value="moondream2">Moondream2 (~4GB VRAM)</option>
<option value="smolvlm">SmolVLM (~4GB VRAM)</option>
<option value="qwen25vl">Qwen2.5-VL 3B (~6GB VRAM)</option>
<option value="qwen3vl">Qwen3-VL 2B (~4GB VRAM)</option>
</select></div>
<div id="swapNotice" class="swap-notice">⏳ Model swap may take 30-60s on first use</div>
<div><label>💬 Prompt</label><textarea class="prompt" id="prompt" rows="3" placeholder="Describe what is happening in this video..."></textarea></div>
<button class="btn" id="analyzeBtn" disabled>⚡ Analyze Video</button>
<div class="error" id="error"></div></section>
<section class="panel"><h2 class="title">📊 Results</h2>
<div id="empty" class="empty"><div class="icon">🔍</div><p>Upload a video and enter a prompt to begin</p></div>
<div class="loading" id="loading"><div class="dots"><span></span><span></span><span></span></div><p style="font-size:15px;font-weight:500" id="loadMsg">Analyzing frames...</p><p style="font-size:13px;color:var(--muted)">T4 GPU • ~2-3s per frame</p></div>
<div id="resultsMeta"></div><div class="results-list" id="resultsList"></div></section>
</main>

<section class="history" id="historySection">
<div class="hist-header"><h2 class="title">📜 Analysis History</h2><button class="refresh" onclick="loadHistory()">🔄</button></div>
<div class="hist-list" id="histList"><div class="empty"><div class="icon">📂</div><p>No saved analyses yet</p></div></div>
</section>
<footer><p>Powered by <strong>Moondream2 · SmolVLM · Qwen2.5-VL · Qwen3-VL</strong></p></footer>
<script>
const fileInput=document.getElementById('fileInput'),prompt=document.getElementById('prompt'),analyzeBtn=document.getElementById('analyzeBtn'),modelSelect=document.getElementById('modelSelect');
const dropZone=document.getElementById('dropZone'),uploadContent=document.getElementById('uploadContent'),fileSelected=document.getElementById('fileSelected');
const errorEl=document.getElementById('error'),emptyEl=document.getElementById('empty'),loadingEl=document.getElementById('loading'),loadMsg=document.getElementById('loadMsg');
const resultsMeta=document.getElementById('resultsMeta'),resultsList=document.getElementById('resultsList');
const dot=document.getElementById('dot'),statusEl=document.getElementById('status'),modelBadge=document.getElementById('modelBadge'),swapNotice=document.getElementById('swapNotice');
let selectedFile=null,currentLoaded='';
const labels={moondream2:'MOONDREAM2',smolvlm:'SMOLVLM',qwen25vl:'QWEN2.5-VL',qwen3vl:'QWEN3-VL'};
fetch('/api/health').then(r=>r.json()).then(d=>{dot.classList.add('on');statusEl.textContent=d.gpu_name||d.device;currentLoaded=d.current_model||'';}).catch(()=>{statusEl.textContent='Offline';});
modelSelect.addEventListener('change',()=>{const v=modelSelect.value;modelBadge.textContent=labels[v]||v.toUpperCase();swapNotice.style.display=(v!==currentLoaded)?'block':'none';});
function setFile(f){if(!f)return;selectedFile=f;uploadContent.classList.add('hidden');fileSelected.classList.remove('hidden');fileSelected.innerHTML=`<span class="icon">🎬</span><div style="flex:1"><div class="name">${esc(f.name)}</div><div class="size">${(f.size/1e6).toFixed(1)} MB</div></div><button class="file-remove" onclick="event.stopPropagation();clearFile()">✕</button>`;updateBtn();}
function clearFile(){selectedFile=null;fileInput.value='';uploadContent.classList.remove('hidden');fileSelected.classList.add('hidden');updateBtn();}
function updateBtn(){analyzeBtn.disabled=!(selectedFile&&prompt.value.trim());}
function esc(s){const d=document.createElement('div');d.textContent=s;return d.innerHTML;}
fileInput.addEventListener('change',e=>{if(e.target.files[0])setFile(e.target.files[0]);});
prompt.addEventListener('input',updateBtn);
dropZone.addEventListener('dragover',e=>{e.preventDefault();dropZone.style.borderColor='var(--accent)';});
dropZone.addEventListener('dragleave',()=>{dropZone.style.borderColor='';});
dropZone.addEventListener('drop',e=>{e.preventDefault();dropZone.style.borderColor='';if(e.dataTransfer.files[0])setFile(e.dataTransfer.files[0]);});
analyzeBtn.addEventListener('click',async()=>{
  errorEl.style.display='none';emptyEl.classList.add('hidden');loadingEl.style.display='block';swapNotice.style.display='none';
  resultsMeta.innerHTML='';resultsList.innerHTML='';analyzeBtn.disabled=true;
  const m=modelSelect.value;if(m!==currentLoaded)loadMsg.textContent='Loading model + analyzing...';
  else loadMsg.textContent='Analyzing frames...';
  const fd=new FormData();fd.append('video',selectedFile);fd.append('prompt',prompt.value);fd.append('model',m);
  try{const r=await fetch('/api/analyze',{method:'POST',body:fd});const d=await r.json();if(!r.ok)throw new Error(d.detail||'Failed');currentLoaded=m;swapNotice.style.display='none';displayResults(d);}
  catch(e){errorEl.textContent=e.message;errorEl.style.display='block';emptyEl.classList.remove('hidden');}
  finally{loadingEl.style.display='none';analyzeBtn.disabled=false;updateBtn();}});
function displayResults(d){
  emptyEl.classList.add('hidden');loadingEl.style.display='none';
  resultsMeta.innerHTML=`<div class="meta"><div class="meta-item"><span class="meta-label">Model</span><span class="meta-val" style="font-size:13px">${labels[d.model]||d.model}</span></div><div class="meta-item"><span class="meta-label">Duration</span><span class="meta-val">${d.video_duration_seconds||0}s</span></div><div class="meta-item"><span class="meta-label">Frames</span><span class="meta-val">${d.frames_analyzed}</span></div><div class="meta-item"><span class="meta-label">Time</span><span class="meta-val">${d.processing_time_seconds}s</span></div></div>`;
  resultsList.innerHTML=d.results.map(r=>`<div class="result-card"><span class="ts">${r.timestamp}</span><span class="desc">${esc(r.description)}</span></div>`).join('');}

async function loadHistory(){try{const r=await fetch('/api/history');const d=await r.json();const a=d.analyses||[];if(!a.length){histList.innerHTML='<div class="empty"><div class="icon">📂</div><p>No saved analyses yet</p></div>';return;}histList.innerHTML=a.map(h=>`<div class="hist-card" onclick="viewHist('${h.id}')"><div class="hist-top"><span class="hist-name">🎬 ${esc(h.video_filename||'')}</span><button class="hist-del" onclick="event.stopPropagation();delHist('${h.id}')">🗑️</button></div><div class="hist-prompt">${esc(h.prompt||'')}</div><div class="hist-meta"><span>${h.frames_analyzed} frames</span><span>${h.processing_time_seconds}s</span><span>${(h.device||'').toUpperCase()}</span></div></div>`).join('');}catch(e){console.error(e);}}
async function viewHist(id){try{const r=await fetch(`/api/history/${id}`);const d=await r.json();displayResults(d);resultsMeta.scrollIntoView({behavior:'smooth'});}catch(e){errorEl.textContent='Failed to load';errorEl.style.display='block';}}
async function delHist(id){if(!confirm('Delete?'))return;try{await fetch(`/api/history/${id}`,{method:'DELETE'});loadHistory();}catch(e){}}
loadHistory();
</script></body></html>
"""
print('✅ UI template loaded')

In [ ]:
# ── 5B-3. Server + ngrok ─────────────────────────────────────
import os, uuid, time, threading, subprocess
from datetime import datetime
from fastapi import FastAPI, File, UploadFile, Form, HTTPException
from fastapi.responses import HTMLResponse
from fastapi.middleware.cors import CORSMiddleware
import uvicorn

# Kill old server/tunnels
try: subprocess.run(['fuser','-k','8000/tcp'],capture_output=True); time.sleep(1)
except: pass
try:
    from pyngrok import ngrok as _ng
    for t in _ng.get_tunnels(): _ng.disconnect(t.public_url)
except: pass

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

MAX_SIZE = 20 * 1024 * 1024
HISTORY = []  # In-memory history

@app.get('/api/history')
async def get_history():
    summaries = [{k: v for k, v in h.items() if k != 'results'} for h in HISTORY]
    return {'analyses': summaries}

@app.get('/api/history/{analysis_id}')
async def get_history_item(analysis_id: str):
    for h in HISTORY:
        if h['id'] == analysis_id:
            return h
    raise HTTPException(404, 'Not found')

@app.delete('/api/history/{analysis_id}')
async def delete_history_item(analysis_id: str):
    for i, h in enumerate(HISTORY):
        if h['id'] == analysis_id:
            HISTORY.pop(i)
            return {'success': True}
    raise HTTPException(404, 'Not found')


@app.get('/api/health')
async def health():
    return {'status':'ok','device':'cuda:0','current_model':manager.current,
            'gpu_name':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}

@app.post('/api/analyze')
async def api_analyze(video:UploadFile=File(...), prompt:str=Form(...), model:str=Form('moondream2')):
    t0 = time.time()
    data = await video.read()
    if len(data)>20*1024*1024: raise HTTPException(413,'Max 20MB')
    ext = os.path.splitext(video.filename or '.mp4')[1]
    path = f'/tmp/{uuid.uuid4().hex}{ext}'
    try:
        with open(path,'wb') as f: f.write(data)
        cap = cv2.VideoCapture(path)
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        dur = total/fps if total>0 and fps>0 else 0
        cap.release()
        results = analyze_video(path, prompt, model_name=model)
        record = {'success':True,'id':uuid.uuid4().hex[:12],'model':model,'prompt':prompt,
                'video_filename':video.filename,'video_duration_seconds':round(dur,2),
                'frames_analyzed':len(results),'processing_time_seconds':round(time.time()-t0,2),
                'device':'cuda:0','results':results, 'created_at':datetime.now().isoformat()}
        HISTORY.insert(0, record)
        return record
    finally:
        if os.path.exists(path): os.remove(path)

@app.get('/',response_class=HTMLResponse)
async def serve_ui(): return FULL_HTML

threading.Thread(target=lambda:uvicorn.run(app,host='0.0.0.0',port=8000,log_level='warning'),daemon=True).start()
time.sleep(3)

import urllib.request
try:
    urllib.request.urlopen('http://localhost:8000/api/health',timeout=5)
    print('✅ Server running')
except Exception as e:
    print(f'❌ Server failed: {e}'); raise

if NGROK_AUTH_TOKEN:
    tunnel = ngrok.connect(8000)
    url = tunnel.public_url
    print(f"\n{'='*50}")
    print(f'🌐 WEB UI: {url}')
    print(f"{'='*50}")
    print(f"⚠️ Click 'Visit Site' if you see an ngrok warning")
else:
    print('⚠️ Set NGROK_AUTH_TOKEN to get a public URL')